# GradientForest

> GradientForest

TODO: Add documentation

In [ ]:
#| default_exp GradientForest

In [ ]:
#| export
from fastcore.utils import *
import statsmodels.api as sm
from nbdev.showdoc import *
from scipy.linalg import svd, inv
from scipy.stats import f
from TracyWidom import TracyWidom
from numba import njit, jit
from collections import namedtuple
# Rpy2
from rpy2.robjects.packages import importr
from rpy2.robjects import default_converter
from rpy2.robjects import numpy2ri
from rpy2.robjects.packages import PackageNotInstalledError
import rpy2.robjects as ro
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd

In [ ]:
#| export
class GradientForestGO:
    "Gradient forest genomic offset statistic."
    def __init__(self, 
                 ntrees: int, # Number of  trees
                ):
        self.ntrees = ntrees
        self._gf = None
        self.F = []
        self._predictors = None
        self._base = importr('base')
        try:
            # Try loading the gradientForest package
            self._gradient_forest = importr('gradientForest')
        except PackageNotInstalledError:
            print("gradientForest package is not installed. You can install it using:")
            print('install.packages("gradientForest", repos="http://R-Forge.R-project.org")')
        except Exception as e:
            print(f"An error occurred: {e}")
    def __str__(self):
        return f"Gradient Forest genomic offset with {self.ntrees} trees."
    __repr__ = __str__

In [ ]:
model = GradientForestGO(ntrees=100)

In [ ]:
#|export
@patch
def fit(self:GradientForestGO,
        Y: np.ndarray, # Allele frequency matrix (nxL)
        X: np.ndarray): # Environmental matrix (nxP)
    "Fits the Geometric genomic offset model. "
    n1, L = Y.shape
    n2, P = X.shape
    if n1 != n2: 
        raise ValueError("Dimensions of array don't match")
    np_cv_rules = numpy2ri.converter+default_converter
    # Use the converter within this context
    with np_cv_rules.context():
        Y_r = ro.r.matrix(np.asarray(Y), nrow=Y.shape[0])
        X_r = ro.r.matrix(np.asarray(X), nrow=X.shape[0])
        df = self._base.as_data_frame(ro.r.cbind(X_r, Y_r))
        cols = self._base.colnames(df)
        predictor_vars = cols[:X.shape[1]]
        self._gf = self._gradient_forest.gradientForest(
            df, predictor_vars=predictor_vars,
            response_vars=cols[X.shape[1]:], ntree=500
        )
        self.F = []
        for col in predictor_vars:
            dist = self._gradient_forest.cumimp_gradientForest(self._gf, col)
            dist = (np.array(dist[0]), np.array(dist[1]))
            self.F.append(dist)

The `fit()` method expects as input an allele matrix $\mathbf Y$ and an environmental matrix $\mathbf X$ with as many rows as individuals. For now, let us simulate them under the ¿generative model?: 

In [ ]:
def generative_model(rng, N, L, P, n_targets):
    X = rng.normal(size=N)
    B = np.zeros(L)
    target_indices = rng.choice(L, n_targets, replace=False)
    B[target_indices] = rng.uniform(-10, 10, size=n_targets)
    U = np.dot(X.reshape(-1, 1), np.array([[-1, 0.5, 1.5]])) + rng.normal(size=(N, 3))
    V = rng.normal(size=(3, L))  # V should have 3 rows to match the columns of U
    Y = np.dot(X.reshape(-1, 1), B.reshape(1, -1)) + np.dot(U, V) + rng.normal(scale=0.5, size=(N, L))
    Y = (Y > 0).astype(int)
    X = np.hstack((X.reshape(-1, 1),  rng.normal(size=(N, P-1))))
    assert X.shape == (N, P)
    assert Y.shape == (N, L)
    return Y, X
rng = np.random.default_rng(1000)
Y, X = generative_model(rng, N=100, L=500, P=10, n_targets=10)

In [ ]:
indices = rng.permutation(100)
training_idx, test_idx = indices[:60], indices[60:]
Xtrain, Xpredict = X[training_idx,:], X[test_idx,:]
Ytrain, Ypredict = Y[training_idx,:], Y[test_idx,:]
model.fit(Ytrain, Xtrain)

Finally, we can predict the genomic offset under two different environments: 

In [ ]:
#| hide
@jit
def _genomic_offset_helper(F, X, Xstar):
    current_F, future_F = np.zeros(X.shape), np.zeros(X.shape)
    for feature_idx in range(X.shape[1]):
        current_F[:, feature_idx] = np.interp(
            X[:, feature_idx],
            F[feature_idx][0],
            F[feature_idx][1]
        )
        future_F[:, feature_idx] = np.interp(
            Xstar[:, feature_idx],
            F[feature_idx][0],
            F[feature_idx][1]
        )
    return current_F-future_F

In [ ]:
#| export 
@patch
def genomic_offset(self:GradientForestGO,
        X: np.ndarray, # Environmental matrix (nxP)
        Xstar: np.ndarray, # Altered environmental matrix (nxP)
           )-> np.ndarray: # A vector of genomic offsets (n)
    "Calculates the genomic offset statistic. " 
    if X.shape != Xstar.shape: 
        raise ValueError("Dimensions of array don't match")
    # Interpolate for each datapoint
    diff = _genomic_offset_helper(self.F, X, Xstar)
    return np.linalg.norm(diff, ord=2, axis=1)


As expected, the genomic offset is zero if both environmental matrixes are identical: 

In [ ]:
offset = model.genomic_offset(Xpredict, Xpredict)
offset

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0.])

In [ ]:
offset = model.genomic_offset(Xpredict, Xpredict+rng.normal(size=Xpredict.shape))
offset

array([0.18361645, 0.01388196, 0.22667139, 0.02739474, 0.00115051,
       0.01398597, 0.00802268, 0.23722511, 0.23463455, 0.00888398,
       0.22688352, 0.18607784, 0.00972496, 0.01553172, 0.20280644,
       0.00247204, 0.01598696, 0.00304033, 0.02735806, 0.17776314,
       0.20591512, 0.20492466, 0.05035983, 0.00234885, 0.00073507,
       0.00538123, 0.01542525, 0.00179949, 0.13238307, 0.23063683,
       0.00592212, 0.01576689, 0.01079999, 0.06972746, 0.04657568,
       0.12715696, 0.01519687, 0.00209921, 0.00314659, 0.0063228 ])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()